# 5 - Sentiment Analysis TextBlob



## 1. Install dan setup

In [1]:
!pip install -q pandas numpy textblob openpyxl tqdm

# =========================================================
# Helper: input fleksibel untuk Google Colab
# =========================================================
from pathlib import Path
import os, re, shutil, json, warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

BASE_DIR = Path("/content")
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "outputs"
EDA_DIR = OUTPUT_DIR / "eda"
SENTIMENT_DIR = OUTPUT_DIR / "sentiment"
MODEL_DIR = OUTPUT_DIR / "models"
FINAL_DIR = OUTPUT_DIR / "final"

for d in [DATA_DIR, OUTPUT_DIR, EDA_DIR, SENTIMENT_DIR, MODEL_DIR, FINAL_DIR]:
    d.mkdir(parents=True, exist_ok=True)


def upload_file_colab(prompt="Upload file CSV"):
    """Upload file di Google Colab dan return Path file pertama."""
    from google.colab import files
    print(prompt)
    uploaded = files.upload()
    if not uploaded:
        raise FileNotFoundError("Tidak ada file yang di-upload.")
    filename = list(uploaded.keys())[0]
    return Path(filename)


def find_or_upload(candidates, upload_prompt):
    """
    Cari file dari daftar kandidat. Kalau tidak ada, minta upload.
    Cocok untuk Colab: bisa lanjut dari file hasil notebook sebelumnya.
    """
    for p in candidates:
        p = Path(p)
        if p.exists():
            print(f"File ditemukan: {p}")
            return p
    return upload_file_colab(upload_prompt)


def normalize_sentiment_label(label):
    """Samakan label menjadi Positive / Neutral / Negative."""
    if pd.isna(label):
        return np.nan
    text = str(label).strip().lower()

    positive_values = {"positive", "positif", "pos", "1", "label_0"}
    neutral_values = {"neutral", "netral", "neu", "0", "label_1"}
    negative_values = {"negative", "negatif", "neg", "-1", "label_2"}

    if text in positive_values:
        return "Positive"
    if text in neutral_values:
        return "Neutral"
    if text in negative_values:
        return "Negative"
    return np.nan


def standardize_dataframe(df):
    """
    Standarisasi kolom dari hasil preprocessing 2a.
    File sumber biasanya punya: Link, Judul, Isi Berita, Status, Tag, Sentimen, Penerbit,
    final_text_clean, final_text_stemmed, jumlah_token_before_sw, jumlah_token_after_sw.
    """
    df = df.copy()

    rename_map = {
        "Link": "source_url",
        "Judul": "title",
        "Isi Berita": "content",
        "Status": "scrape_status",
        "Tag": "tag",
        "Sentimen": "sentiment_original",
        "Sentiment": "sentiment_original",
        "Penerbit": "publisher",
        "Tanggal": "published_date"
    }

    for old, new in rename_map.items():
        if old in df.columns and new not in df.columns:
            df[new] = df[old]

    # Label asli/manual untuk evaluasi
    if "label_true" not in df.columns:
        if "sentiment_original" in df.columns:
            df["label_true"] = df["sentiment_original"].apply(normalize_sentiment_label)
        else:
            df["label_true"] = np.nan

    # Kolom teks fallback
    if "final_text_clean" not in df.columns:
        if "content" in df.columns:
            df["final_text_clean"] = df["content"].fillna("").astype(str).str.lower()
        else:
            raise KeyError("Tidak ditemukan kolom final_text_clean atau content/Isi Berita.")

    if "final_text_stemmed" not in df.columns:
        df["final_text_stemmed"] = df["final_text_clean"]

    # Token count fallback
    if "jumlah_token_before_sw" not in df.columns:
        df["jumlah_token_before_sw"] = df["final_text_clean"].fillna("").astype(str).str.split().apply(len)
    if "jumlah_token_after_sw" not in df.columns:
        df["jumlah_token_after_sw"] = df["final_text_stemmed"].fillna("").astype(str).str.split().apply(len)

    # Filter success kalau kolom status ada
    if "scrape_status" in df.columns:
        success_mask = df["scrape_status"].astype(str).str.lower().eq("success")
        if success_mask.sum() > 0:
            df = df[success_mask].copy()

    # Buang teks kosong
    df["final_text_clean"] = df["final_text_clean"].fillna("").astype(str)
    df["final_text_stemmed"] = df["final_text_stemmed"].fillna("").astype(str)
    df = df[df["final_text_clean"].str.strip().ne("")].copy()
    df = df.reset_index(drop=True)

    return df


def load_after_2a_csv(path):
    """Load CSV hasil preprocessing 2a dengan fallback encoding."""
    path = Path(path)
    try:
        df = pd.read_csv(path)
    except UnicodeDecodeError:
        df = pd.read_csv(path, encoding="latin-1")
    return standardize_dataframe(df)


def safe_display(obj, n=5):
    try:
        display(obj)
    except Exception:
        print(obj.head(n) if hasattr(obj, "head") else obj)


## 2. Input data
Gunakan file standar dari EDA jika runtime masih sama, atau upload `hasil_preprocessing_pertamina.csv`.

In [2]:

INPUT_PATH = find_or_upload(
    candidates=[
        DATA_DIR / "hasil_preprocessing_pertamina_standardized.csv",
        "/content/hasil_preprocessing_pertamina.csv",
        "/content/data/hasil_preprocessing_pertamina.csv",
        "/content/data/processed/hasil_preprocessing_pertamina.csv",
    ],
    upload_prompt="Upload file hasil_preprocessing_pertamina.csv atau hasil_preprocessing_pertamina_standardized.csv"
)

df = load_after_2a_csv(INPUT_PATH)
print("Input:", INPUT_PATH)
print("Shape:", df.shape)
safe_display(df.head(3))


Upload file hasil_preprocessing_pertamina.csv atau hasil_preprocessing_pertamina_standardized.csv


Saving hasil_preprocessing_pertamina.csv to hasil_preprocessing_pertamina.csv
Input: hasil_preprocessing_pertamina.csv
Shape: (1829, 19)


,Link,Judul,Isi Berita,Status,Tag,Sentimen,Penerbit,final_text_clean,final_text_stemmed,jumlah_token_before_sw,jumlah_token_after_sw,source_url,title,content,scrape_status,tag,sentiment_original,publisher,label_true
0,https://money.kompas.com/image/2017/10/03/1815...,Foto : CSR Pertamina Lubricants Kini Fokus ke ...,Diskusi mengenai CSR di industri pelumas berta...,success,Ekonomi,Neutral,Kompas,diskusi mengenai csr industri pelumas bertajuk...,diskusi kena csr industri lumas tajuk cari for...,16,13,https://money.kompas.com/image/2017/10/03/1815...,Foto : CSR Pertamina Lubricants Kini Fokus ke ...,Diskusi mengenai CSR di industri pelumas berta...,success,Ekonomi,Neutral,Kompas,Neutral
1,https://money.kompas.com/read/2016/06/06/14280...,"Gandeng Dua Bank, Pertamina Lubricants Yakin P...","JAKARTA, KOMPAS.com - Anak perusahaan Pertamin...",success,Kecelakaan Kerja,Positive,Kompas,jakarta kompas com anak perusahaan pertamina y...,anak usaha pertamina pertamina lubricants gand...,130,95,https://money.kompas.com/read/2016/06/06/14280...,"Gandeng Dua Bank, Pertamina Lubricants Yakin P...","JAKARTA, KOMPAS.com - Anak perusahaan Pertamin...",success,Kecelakaan Kerja,Positive,Kompas,Positive
2,https://money.kompas.com/read/2017/02/03/12195...,"Dua Pucuk Pimpinan Pertamina Dicopot, Yenni An...","JAKARTA, KOMPAS.com — Pasca-pencopotan dua puc...",success,Migas,Positive,Kompas,jakarta kompas com pasca pencopotan dua pucuk ...,pasca copot pucuk pimpin pertamina menteri bad...,145,120,https://money.kompas.com/read/2017/02/03/12195...,"Dua Pucuk Pimpinan Pertamina Dicopot, Yenni An...","JAKARTA, KOMPAS.com — Pasca-pencopotan dua puc...",success,Migas,Positive,Kompas,Positive


## 3. TextBlob sentiment

In [3]:

from textblob import TextBlob
from tqdm.auto import tqdm

tqdm.pandas()

# TextBlob tetap dipakai sebagai baseline sederhana.
# Karena artikel berbahasa Indonesia, hasilnya mungkin banyak netral.
TEXT_COL_TEXTBLOB = "final_text_clean"

def get_textblob_polarity(text):
    return TextBlob(str(text)).sentiment.polarity

def get_textblob_subjectivity(text):
    return TextBlob(str(text)).sentiment.subjectivity

def classify_textblob_sentiment(polarity):
    if polarity > 0.05:
        return "Positive"
    elif polarity < -0.05:
        return "Negative"
    return "Neutral"

print("Menjalankan TextBlob pada kolom:", TEXT_COL_TEXTBLOB)
df["textblob_polarity"] = df[TEXT_COL_TEXTBLOB].progress_apply(get_textblob_polarity)
df["textblob_subjectivity"] = df[TEXT_COL_TEXTBLOB].progress_apply(get_textblob_subjectivity)
df["textblob_sentiment"] = df["textblob_polarity"].apply(classify_textblob_sentiment)

summary = df["textblob_sentiment"].value_counts().reindex(["Positive", "Neutral", "Negative"], fill_value=0).reset_index()
summary.columns = ["sentiment", "jumlah"]
safe_display(summary)


Menjalankan TextBlob pada kolom: final_text_clean


  0%|          | 0/1829 [00:00<?, ?it/s]

  0%|          | 0/1829 [00:00<?, ?it/s]

,sentiment,jumlah
0,Positive,513
1,Neutral,1090
2,Negative,226


## 4. Export hasil TextBlob

In [4]:

OUTPUT_CSV = SENTIMENT_DIR / "hasil_textblob_sentiment.csv"
OUTPUT_EXCEL = SENTIMENT_DIR / "hasil_textblob_sentiment.xlsx"

df.to_csv(OUTPUT_CSV, index=False)
with pd.ExcelWriter(OUTPUT_EXCEL, engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="TextBlob_Result", index=False)
    summary.to_excel(writer, sheet_name="Distribution", index=False)

print("CSV tersimpan:", OUTPUT_CSV)
print("Excel tersimpan:", OUTPUT_EXCEL)

from google.colab import files
files.download(str(OUTPUT_CSV))


CSV tersimpan: /content/outputs/sentiment/hasil_textblob_sentiment.csv
Excel tersimpan: /content/outputs/sentiment/hasil_textblob_sentiment.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>